## Data: Sleeper's public API, cached on disk

Every request goes through `get()`, which keeps a gzipped copy under `notebooks/.cache/sleeper/` (git-ignored). All seasons used
here are finished, so a cached response never goes stale. The first run downloads a few thousand small files (a few minutes
under Sleeper's rate limit); later runs read the cache.

**Which leagues.** The owner's own finished league is *goon squad* (2025, 8 teams). It has 8 playoff spots for 8 teams, so every
team makes the playoffs and it says nothing about playoff odds; it still counts for title and flush odds. No other finished
league is reachable from the owner's account: the members of both of the owner's leagues play in no other Sleeper league.
To get enough seasons, the notebook widens the net from two **public example leagues** that the open-source `ffscrapr` R package
uses in its documentation (league ids `522458773317046272` and `521379020332068864`). It takes their members' 2023-2025 leagues and
keeps the ordinary ones:

* finished (`status == complete`), head-to-head (no best ball, no extra game against the league median),
* 8 to 14 teams, fewer playoff spots than teams, playoffs starting in week 14-16,
* offense-only lineups (no IDP slots, which the projections here do not cover).

Only aggregate numbers are reported. Apart from the owner's own league, league names, team names and usernames are never printed; other leagues are shown as a short hash.

In [1]:
import gzip, hashlib, json, time, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd

CACHE = Path('.cache') / 'sleeper'
CACHE.mkdir(parents=True, exist_ok=True)
BASE = 'https://api.sleeper.app'

def get(path):
    """GET a Sleeper path (e.g. '/v1/league/123'), cached forever on disk. Returns None on a 404."""
    f = CACHE / (hashlib.sha1(path.encode()).hexdigest()[:20] + '.json.gz')
    if f.exists():
        with gzip.open(f, 'rt') as fh: return json.load(fh)
    for attempt in range(5):
        try:
            with urllib.request.urlopen(BASE + path, timeout=30) as r:
                data = json.load(r)
            break
        except urllib.error.HTTPError as e:
            if e.code == 404: data = None; break
            time.sleep(1 + 2 * attempt)
        except Exception:
            time.sleep(1 + 2 * attempt)
    else:
        raise RuntimeError('failed: ' + path)
    with gzip.open(f, 'wt') as fh: json.dump(data, fh)
    return data

def get_many(paths, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(get, paths))

anon = lambda lid: hashlib.sha1(str(lid).encode()).hexdigest()[:6]

OWNER_USER = '1270089224720232448'
OWNER_LEAGUE_2025 = '1270085638657413120'           # goon squad
PUBLIC_SEEDS = ['522458773317046272', '521379020332068864']
SEASONS = ['2023', '2024', '2025']
OFFENSE_SLOTS = {'QB', 'RB', 'WR', 'TE', 'K', 'DEF', 'FLEX', 'SUPER_FLEX', 'REC_FLEX', 'WRRB_FLEX'}
NON_LINEUP = {'BN', 'IR', 'TAXI'}

def ordinary(l):
    s = l.get('settings') or {}
    slots = [p for p in l.get('roster_positions') or [] if p not in NON_LINEUP]
    n, P = l.get('total_rosters') or 0, s.get('playoff_teams') or 6
    return (l.get('status') == 'complete' and not s.get('best_ball') and not s.get('league_average_match')
            and 8 <= n <= 14 and P < n and 14 <= (s.get('playoff_week_start') or 15) <= 16
            and all(x in OFFENSE_SLOTS for x in slots) and s.get('type', 0) in (0, 1, 2))

owner_leagues = get(f'/v1/user/{OWNER_USER}/leagues/nfl/2025') or []
members = sorted({u['user_id'] for lid in PUBLIC_SEEDS for u in (get(f'/v1/league/{lid}/users') or [])})
found = {}
for lists in get_many([f'/v1/user/{u}/leagues/nfl/{y}' for u in members for y in SEASONS]):
    for l in lists or []: found[l['league_id']] = l
ext = sorted((l for l in found.values() if ordinary(l)), key=lambda l: (l['season'], l['league_id']))
print('owner 2025 leagues:', [(l['name'], l['total_rosters'], l['settings'].get('playoff_teams')) for l in owner_leagues])
print(f'public-seed members: {len(members)}, their 2023-25 leagues: {len(found)}, ordinary: {len(ext)}')
print(pd.Series([l['season'] for l in ext]).value_counts().sort_index().to_dict())

owner 2025 leagues: [('goon squad', 8, 8)]
public-seed members: 24, their 2023-25 leagues: 177, ordinary: 106
{'2023': 42, '2024': 35, '2025': 29}


In [2]:
def load_league(lid):
    lg = get(f'/v1/league/{lid}')
    s = lg['settings']
    last = 18 if int(lg['season']) >= 2021 else 17
    weeks = list(range(1, last + 1))
    mups = get_many([f'/v1/league/{lid}/matchups/{w}' for w in weeks])
    return dict(lid=lid, season=lg['season'], league=lg, settings=s, slots=[p for p in lg['roster_positions'] if p not in NON_LINEUP],
                rosters=get(f'/v1/league/{lid}/rosters'), matchups={w: m or [] for w, m in zip(weeks, mups)},
                winners=get(f'/v1/league/{lid}/winners_bracket') or [], losers=get(f'/v1/league/{lid}/losers_bracket') or [])

POSITIONS = ['QB', 'RB', 'WR', 'TE', 'K', 'DEF']
def load_season(season):
    q = '&'.join(f'position[]={p}' for p in POSITIONS)
    weeks = list(range(1, 19))
    proj = get_many([f'/projections/nfl/{season}/{w}?season_type=regular&{q}' for w in weeks])
    stats = get_many([f'/v1/stats/nfl/regular/{season}/{w}' for w in weeks])
    scores = get_many([f'/scores/nfl/regular/{season}/{w}' for w in weeks])
    return dict(proj={w: {r['player_id']: r for r in (p or [])} for w, p in zip(weeks, proj)},
                stats={w: s or {} for w, s in zip(weeks, stats)}, scores={w: s or [] for w, s in zip(weeks, scores)})

t0 = time.time()
LEAGUES = [load_league(OWNER_LEAGUE_2025)] + [load_league(l['league_id']) for l in ext]
SEASON_DATA = {s: load_season(s) for s in sorted({L['season'] for L in LEAGUES})}
print(f'loaded {len(LEAGUES)} leagues and {len(SEASON_DATA)} seasons in {time.time() - t0:.0f}s')

loaded 107 leagues and 3 seasons in 1s


## The engine's rules, ported from `src/engine/`

* `make_scorer`: `makeScorer` in `model.ts` (PPR / half / standard column by `rec`, pass-TD points, TE/WR/RB catch bonuses).
  It scores both projections and stat lines, as the app does. Custom scoring beyond that is ignored, also as the app does.
* `played`: a stat line counts when `gms_active` is set, or for a team defense when `gp > 0` (the app's default, not `referenceCompat`).
* `team_sd`: `teamSd` in `model.ts`. Residual SD per position (actual minus projected, players projected above 0.5) on the last
  four completed weeks, with the app's fallbacks under 25 samples; the team SD is the root sum of squares over the starting slots,
  a flex slot counting as a WR.
* `best_total`: `bestSet` in `lineup.ts`, the greedy lineup (fixed slots first, then flex slots from most to least restrictive).
* `win_prob`: `winProb` in `league.ts`: P(A > B) with each score Normal(mean, sd) and independent.

In [3]:
from math import erf, sqrt, log2, ceil

ELIG = {'QB': ('QB',), 'RB': ('RB',), 'WR': ('WR',), 'TE': ('TE',), 'K': ('K',), 'DEF': ('DEF',),
        'FLEX': ('RB', 'WR', 'TE'), 'SUPER_FLEX': ('QB', 'RB', 'WR', 'TE'), 'REC_FLEX': ('WR', 'TE'), 'WRRB_FLEX': ('RB', 'WR')}
FALLBACK_SD = {'QB': 7.9, 'RB': 6.2, 'WR': 6.6, 'TE': 6.4, 'K': 5.5, 'DEF': 6.5}
MIN_CAL_SAMPLES = 25

def scorer_key(sc):
    rec = sc.get('rec', 1) if sc.get('rec') is not None else 1
    base = 'pts_ppr' if rec >= 0.75 else 'pts_half_ppr' if rec >= 0.25 else 'pts_std'
    pass_td = (sc.get('pass_td') if sc.get('pass_td') is not None else 4) - 4
    bonus = tuple((k, sc[k]) for k in ('bonus_rec_te', 'bonus_rec_wr', 'bonus_rec_rb') if sc.get(k))
    return (base, pass_td, bonus)

def make_scorer(key):
    base, pass_td, bonus = key
    def score(st):
        if not st: return 0.0
        v = st[base] if st.get(base) is not None else (st.get('pts_ppr') or 0)
        if pass_td and st.get('pass_td'): v += pass_td * st['pass_td']
        for k, w in bonus:
            if st.get(k): v += w * st[k]
        return v
    return score

def fill_order(slots):
    idx = range(len(slots))
    fixed = [i for i in idx if len(ELIG[slots[i]]) == 1]
    flex = sorted((i for i in idx if len(ELIG[slots[i]]) > 1), key=lambda i: len(ELIG[slots[i]]))
    return [slots[i] for i in fixed + flex]

def best_total(cands, order):
    """cands: list of (value, pos); order: fill_order(slots). Greedy, as bestSet in lineup.ts."""
    cand = sorted(cands, key=lambda c: -c[0])
    used = [False] * len(cand)
    tot = 0.0
    for s in order:
        el = ELIG[s]
        for i, (v, ps) in enumerate(cand):
            if not used[i] and ps in el:
                used[i] = True; tot += v; break
    return tot

def win_prob(a, b, sd):
    return 0.5 * (1 + erf((a - b) / (sqrt(2) * sd) / sqrt(2)))

class Season:
    """One NFL season of Sleeper projections, stats and game scores, with the per-scoring caches the engine needs."""
    def __init__(self, season, d):
        self.season, self.proj, self.stats, self.scores = season, d['proj'], d['stats'], d['scores']
        self.info = {}
        for w in sorted(self.proj, reverse=True):          # latest week that has the player wins, as in model.ts
            for pid, r in self.proj[w].items():
                if pid in self.info: continue
                pl = r.get('player') or {}
                self.info[pid] = (pl.get('position') or ('DEF' if pid.isalpha() else None), pl.get('team') or r.get('team'))
        self.playing = {w: {t for g in self.scores[w] for t in ((g.get('metadata') or {}).get('home_team'), (g.get('metadata') or {}).get('away_team')) if t}
                        for w in self.scores}
        self._pp, self._sd = {}, {}

    def pos(self, pid): return self.info.get(pid, (('DEF' if pid.isalpha() else None), None))[0]
    def team(self, pid): return self.info.get(pid, (None, pid if pid.isalpha() else None))[1]

    def played(self, w, pid):
        s = self.stats.get(w, {}).get(pid)
        return bool(s) and (bool(s.get('gms_active')) or (self.pos(pid) == 'DEF' and (s.get('gp') or 0) > 0))

    def proj_pts(self, key, w):
        """pid -> projected points in week w under scoring `key` (cached)."""
        k = (key, w)
        if k not in self._pp:
            score = make_scorer(key)
            self._pp[k] = {pid: score(r.get('stats')) for pid, r in self.proj.get(w, {}).items()}
        return self._pp[k]

    def actual(self, key, w, pid):
        return make_scorer(key)(self.stats[w][pid]) if self.played(w, pid) else None

    def pos_sd(self, key, cur):
        """Residual SD per position on the app's calibration weeks max(1, cur-4) .. cur-1."""
        k = (key, cur)
        if k not in self._sd:
            score = make_scorer(key)
            resid = {p: [] for p in FALLBACK_SD}
            for w in range(max(1, cur - 4), cur):
                pp = self.proj_pts(key, w)
                for pid, pr in pp.items():
                    pos = self.pos(pid)
                    if pos not in resid or pr <= 0.5 or not self.played(w, pid): continue
                    resid[pos].append(score(self.stats[w][pid]) - pr)
            self._sd[k] = {p: (float(np.std(v, ddof=1)) if len(v) >= MIN_CAL_SAMPLES else FALLBACK_SD[p]) for p, v in resid.items()}
        return self._sd[k]

    def team_sd(self, key, slots, cur):
        sdp = self.pos_sd(key, cur)
        return sqrt(sum((sdp[ELIG[s][0]] if len(ELIG[s]) == 1 else sdp['WR']) ** 2 for s in slots)) or 20.0

SEASONS_ = {s: Season(s, d) for s, d in SEASON_DATA.items()}
for s, S in SEASONS_.items():
    k = scorer_key({'rec': 1})
    print(s, 'PPR position SDs at week 10:', {p: round(v, 1) for p, v in S.pos_sd(k, 10).items()},
          '| team SD, 1QB/2RB/2WR/TE/FLEX/K/DEF:', round(S.team_sd(k, ['QB', 'RB', 'RB', 'WR', 'WR', 'TE', 'FLEX', 'K', 'DEF'], 10), 1))

2023 PPR position SDs at week 10: {'QB': 7.2, 'RB': 5.1, 'WR': 5.4, 'TE': 5.3, 'K': 4.6, 'DEF': 5.7} | team SD, 1QB/2RB/2WR/TE/FLEX/K/DEF: 16.5
2024 PPR position SDs at week 10: {'QB': 6.8, 'RB': 6.0, 'WR': 6.1, 'TE': 4.9, 'K': 4.7, 'DEF': 5.3} | team SD, 1QB/2RB/2WR/TE/FLEX/K/DEF: 17.4
2025 PPR position SDs at week 10: {'QB': 7.7, 'RB': 6.4, 'WR': 5.9, 'TE': 5.4, 'K': 4.6, 'DEF': 6.2} | team SD, 1QB/2RB/2WR/TE/FLEX/K/DEF: 18.2


# Do starters' scores move together? Measuring correlation in fantasy scoring

**The question.** The app's season simulation and pregame win chances treat every starter's score as independent of every other
starter's. Each player's miss (actual minus Sleeper's projection) has a typical size by position (the *positional residual SD*, about 5 to 8
points), and a team's weekly spread is the square root of the sum of the squared spreads of its starting slots (`teamSd` in
`src/engine/model.ts`; about 17 to 19 points). If quarterback and receiver tend to boom or bust *together*, that is too small: variances of
positively correlated scores add up to more than the sum of the parts. This notebook measures how much, and what to change.

**Plan.**
1. Build a table of every player-week residual, 2023-2025 (PPR).
2. Measure correlations by role: same NFL team, opposing teams in one game, and unrelated games.
3. Check the real lineups of the 107 league-seasons: how big are team-score misses, compared with the app's team SD, and where does any extra spread come from?
4. Prototype a correction and check that it improves pregame win chances.
5. Say what to change.

Only aggregate numbers are shown; leagues are shown as short hashes at most.

In [4]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
from scipy import stats as sps
from sklearn.linear_model import LogisticRegression

OUT = Path('..') / 'docs' / 'validation'
OUT.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(2026)
KEY = scorer_key({'rec': 1})                 # PPR, 4-point pass TD: the key the app uses for plain PPR
t_start = time.time()
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

def corr_ci(x, y):
    """Pearson r with a 95% Fisher-z interval on the pairs where both are present."""
    m = ~(np.isnan(x) | np.isnan(y)); x, y = x[m], y[m]; n = len(x)
    if n < 10: return dict(r=np.nan, lo=np.nan, hi=np.nan, n=n)
    r = float(np.corrcoef(x, y)[0, 1]); se = 1 / sqrt(n - 3); z = np.arctanh(r)
    return dict(r=r, lo=float(np.tanh(z - 1.96 * se)), hi=float(np.tanh(z + 1.96 * se)), n=n)

## 1. Every player-week miss (the residual table)

For each regular-season week (1 to 18) and each player Sleeper projected above 0.5 PPR points *who played*, the **residual** is actual points
minus projected points. The **standardized residual** divides it by that position's residual SD in that season, so a quarterback's and a kicker's
misses are on one scale (about 1 means "a typical miss"). The NFL team and opponent come from the projection record, falling back to the game
pairing in the NFL scores feed. Within an NFL team and week, players are ranked by projection inside their position: `QB1`, `RB1`, `RB2`, `WR1`-`WR3`, `TE1`, `K`, `DEF`
(the defense is the team itself). That rank is by *pregame projection* among everyone projected above 0.5, so a WR1 is who Sleeper expected to
be the top receiver, not who happened to score the most.

In [5]:
def role_of(pos, rk):
    if pos in ('K', 'DEF'): return pos if rk == 1 else pos + 'x'
    return f'{pos}{rk}' if rk <= {'QB': 1, 'RB': 2, 'WR': 3, 'TE': 1}[pos] else pos + 'x'

def build_players():
    rows = []
    score = make_scorer(KEY)
    for season, S in SEASONS_.items():
        for w in range(1, 19):
            if not S.proj.get(w): continue
            pair = {}
            for g in S.scores[w]:
                m = g.get('metadata') or {}
                if m.get('home_team') and m.get('away_team'):
                    pair[m['home_team']] = m['away_team']; pair[m['away_team']] = m['home_team']
            pp = S.proj_pts(KEY, w)
            for pid, r in S.proj[w].items():
                pos = S.pos(pid)
                if pos not in POSITIONS or pp[pid] <= 0.5: continue
                team = r.get('team') or (r.get('player') or {}).get('team') or (pid if pid.isalpha() else None)
                opp = r.get('opponent') or pair.get(team)
                pl = S.played(w, pid)
                rows.append((season, w, pid, pos, team, opp, pp[pid], pl, score(S.stats[w][pid]) if pl else np.nan))
    d = pd.DataFrame(rows, columns=['season', 'week', 'pid', 'pos', 'team', 'opp', 'proj', 'played', 'actual'])
    d['game'] = d.season + '-' + d.week.astype(str) + '-' + d[['team', 'opp']].fillna('?').apply(lambda r: '|'.join(sorted(r)), axis=1)
    d['rank'] = d.groupby(['season', 'week', 'team', 'pos']).proj.rank(method='first', ascending=False).astype(int)
    d['role'] = [role_of(p, k) for p, k in zip(d.pos, d['rank'])]
    d['resid'] = d.actual - d.proj
    return d

PW_ALL = build_players()
PW = PW_ALL[PW_ALL.played].copy()
POS_SD = PW.groupby(['season', 'pos']).resid.std().unstack()         # season x position residual SD (all weeks)
PW['z'] = PW.resid / [POS_SD.loc[s, p] for s, p in zip(PW.season, PW.pos)]
print(f"player-weeks projected > 0.5: {len(PW_ALL):,}; of those that played: {len(PW):,} ({PW.played.mean():.1%} of the table); missing opponent: {PW.opp.isna().sum()}")
print('\nResidual SD by season and position (points, PPR, all weeks):'); print(POS_SD.round(2))
summ = PW.groupby('pos').agg(n=('resid', 'size'), mean_proj=('proj', 'mean'), mean_resid=('resid', 'mean'), sd_resid=('resid', 'std'))
print('\nAll seasons by position:'); print(summ.round(2))
print('\nStandardized residual: mean %.3f, SD %.3f (should be about 0 and 1)' % (PW.z.mean(), PW.z.std()))

player-weeks projected > 0.5: 20,152; of those that played: 20,147 (100.0% of the table); missing opponent: 0

Residual SD by season and position (points, PPR, all weeks):
pos      DEF     K    QB    RB    TE    WR
season                                    
2023    6.29  4.62  7.17  5.78  4.69  6.15
2024    5.68  4.92  7.45  5.78  4.77  6.18
2025    5.91  4.75  7.59  6.06  4.92  5.86

All seasons by position:
        n  mean_proj  mean_resid  sd_resid
pos                                       
DEF  1632       6.54        0.49      5.98
K    1606       7.98        0.04      4.77
QB   1653      18.66       -2.26      7.42
RB   4374       7.82        0.16      5.88
TE   3763       4.94        0.25      4.80
WR   7119       7.37       -0.08      6.07

Standardized residual: mean -0.007, SD 1.005 (should be about 0 and 1)


## 2. Who moves with whom? Correlation by role

For each pair of roles the correlation is taken across NFL team-weeks where both players played (n is the number of such team-weeks), on
the standardized residuals; the raw-point version is in the last column and agrees closely. The bars are 95% confidence intervals
(Fisher z). Rough guide: |r| of 0.1 is small; 0.3 is a clear link.

* **Same NFL team**: do a team's players have good and bad days together (a shootout lifts everyone)?
* **Opposing teams in one game**: does one side's boom come at the other's expense, or do both sides of a shootout rise together?
* **Baseline**: random pairs of players from *different* games in the same week. Any correlation here is a league-wide weekly factor
  (weather, a week where projections run low everywhere), not a football relationship. Its interval comes from resampling whole weeks.

In [6]:
ROLES = ['QB1', 'RB1', 'RB2', 'WR1', 'WR2', 'WR3', 'TE1', 'K', 'DEF']
TW_KEY = ['season', 'week', 'team']
WZ = PW[PW.role.isin(ROLES)].pivot_table(index=TW_KEY, columns='role', values='z', aggfunc='first').reindex(columns=ROLES)
WR_ = PW[PW.role.isin(ROLES)].pivot_table(index=TW_KEY, columns='role', values='resid', aggfunc='first').reindex(columns=ROLES)
OPP = PW.drop_duplicates(TW_KEY).set_index(TW_KEY).opp                 # (season, week, team) -> opponent
opp_idx = pd.MultiIndex.from_arrays([WZ.index.get_level_values(0), WZ.index.get_level_values(1), OPP.reindex(WZ.index).values])
WZ_O = WZ.reindex(opp_idx); WZ_O.index = WZ.index                      # the opposing team's row, aligned to ours
WR_O = WR_.reindex(opp_idx); WR_O.index = WR_.index
first_of_pair = pd.Series(WZ.index.get_level_values('team') < OPP.reindex(WZ.index).values, index=WZ.index)

def role_pair(a, b, kind):
    if kind == 'same':
        x, y, xr, yr = WZ[a], WZ[b], WR_[a], WR_[b]
    else:
        x, y, xr, yr = WZ[a], WZ_O[b], WR_[a], WR_O[b]
        if a == b: x, y, xr, yr = x[first_of_pair], y[first_of_pair], xr[first_of_pair], yr[first_of_pair]   # count each game once
    out = corr_ci(x.values, y.values)
    out['r_raw'] = corr_ci(xr.values, yr.values)['r']
    return out

PAIRS = [('same team', 'QB1', 'WR1', 'same'), ('same team', 'QB1', 'WR2', 'same'), ('same team', 'QB1', 'TE1', 'same'),
         ('same team', 'QB1', 'RB1', 'same'), ('same team', 'WR1', 'WR2', 'same'), ('same team', 'RB1', 'RB2', 'same'),
         ('same team', 'RB1', 'WR1', 'same'), ('same team', 'QB1', 'K', 'same'), ('same team', 'QB1', 'DEF', 'same'),
         ('same team', 'K', 'DEF', 'same'),
         ('opposing', 'QB1', 'QB1', 'opp'), ('opposing', 'WR1', 'WR1', 'opp'), ('opposing', 'QB1', 'DEF', 'opp'),
         ('opposing', 'RB1', 'DEF', 'opp'), ('opposing', 'K', 'DEF', 'opp')]
rows = []
for grp, a, b, kind in PAIRS:
    c = role_pair(a, b, kind)
    rows.append(dict(group=grp, pair=f'{a} - {b}' if kind == 'same' else f'{a} - opp {b}', **c))

# baseline: random pairs from different games in the same week; per-week sums allow a bootstrap over weeks
def baseline_sums(n_pairs=20000):
    sums = {}
    for (s, w), g in PW.groupby(['season', 'week']):
        z, gm = g.z.values, g.game.values
        i, j = RNG.integers(0, len(g), n_pairs), RNG.integers(0, len(g), n_pairs)
        k = gm[i] != gm[j]; x, y = z[i[k]], z[j[k]]
        sums[(s, w)] = np.array([2 * k.sum(), (x + y).sum(), (x * x + y * y).sum(), 2 * (x * y).sum()])   # both orders: n, sum, sum sq, cross
    return sums
def corr_from_sums(a):
    n, s, ss, sxy = a
    m = s / n
    return (sxy / n - m * m) / (ss / n - m * m)
BASE = baseline_sums()
def base_corr(keys): return corr_from_sums(sum(BASE[k] for k in keys))
bkeys = list(BASE); r_base = base_corr(bkeys)
boot = [base_corr([bkeys[i] for i in RNG.integers(0, len(bkeys), len(bkeys))]) for _ in range(500)]
rows.append(dict(group='baseline', pair='different games, same week', r=r_base, lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5),
                 n=int(sum(v[0] for v in BASE.values()) // 2), r_raw=np.nan))
PT = pd.DataFrame(rows)
print(PT.assign(r=PT.r.round(3), lo=PT.lo.round(3), hi=PT.hi.round(3), r_raw=PT.r_raw.round(3)).to_string(index=False))
RHO_WEEK = r_base

# chart
fig, ax = plt.subplots(figsize=(8.5, 7))
col = {'same team': '#2a6f97', 'opposing': '#c8553d', 'baseline': '#7a7a7a'}
y = np.arange(len(PT))[::-1]
for yi, (_, r) in zip(y, PT.iterrows()):
    ax.errorbar(r.r, yi, xerr=[[r.r - r.lo], [r.hi - r.r]], fmt='o', color=col[r.group], capsize=3, ms=6)
    ax.text(0.5, yi, f'n={r.n:,}' + (' pairs' if r.group == 'baseline' else ''), va='center', fontsize=8, color='#555')
ax.set_yticks(y); ax.set_yticklabels(PT.pair); ax.axvline(0, color='k', lw=0.8); ax.set_xlim(-0.45, 0.78)
ax.set_xlabel('correlation of standardized residuals (95% CI)')
ax.set_title('Do fantasy scores move together? (PPR, 2023-2025, weeks 1-18)', loc='left', fontsize=11)
for g, c in col.items(): ax.plot([], [], 'o', color=c, label=g)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=3, frameon=False); ax.grid(axis='x', alpha=0.25)
plt.tight_layout(); plt.savefig(OUT / 'corr_pairs.png', dpi=140); plt.close()

    group                       pair      r     lo     hi       n  r_raw
same team                  QB1 - WR1  0.371  0.328  0.412    1632  0.371
same team                  QB1 - WR2  0.349  0.306  0.391    1632  0.349
same team                  QB1 - TE1  0.295  0.250  0.339    1632  0.296
same team                  QB1 - RB1  0.089  0.040  0.137    1632  0.089
same team                  WR1 - WR2  0.017 -0.031  0.066    1632  0.017
same team                  RB1 - RB2 -0.028 -0.076  0.021    1622 -0.030
same team                  RB1 - WR1  0.029 -0.020  0.077    1632  0.028
same team                    QB1 - K  0.044 -0.005  0.093    1606  0.043
same team                  QB1 - DEF -0.085 -0.133 -0.037    1632 -0.086
same team                    K - DEF  0.241  0.194  0.287    1606  0.240
 opposing              QB1 - opp QB1  0.180  0.113  0.246     816  0.182
 opposing              WR1 - opp WR1  0.061 -0.008  0.129     816  0.059
 opposing              QB1 - opp DEF -0.406 -0.446 

## 3. Real lineups: is the team score as spread out as the app thinks?

For every league-season, every regular-season week before the playoffs, every team: sum the **starters' actual points** (the league's own scoring,
`starters_points`) and subtract the sum of the starters' **projected points** (Sleeper projection scored with the league's settings, as the app
does; a starter with no projection counts as 0). That is the team's miss. The app claims its misses have a spread equal to `team_sd` (rolling
four-week fit, as in the app). So **z = miss / team_sd should have a spread of 1**. If it is above 1, the app is too sure of itself.

How starters are classified in the lineup table:
* `played`: has a stat line with games active.
* `bye`: his NFL team had no game that week (projection 0, score 0, harmless).
* `dnp`: his team played but he did not (inactive, injured, scratched). **Data problem to keep in mind:** Sleeper's weekly projection file, as downloaded
  now, keeps almost nobody who did not play (of players projected above 0.5 PPR points, 99.98% played). So these starters have a projection of 0 in the file and their
  "miss" is 0, which hides the cost of a start who sits. As a *pregame-view* sensitivity, a `dnp` starter is given the last projection he had in the
  previous four weeks (what the app would have shown before kickoff).
* `empty`: an empty lineup slot (pid `0`).

In [7]:
PAIR_CACHE, TEAM_HIST = {}, {}
def nfl_pair(S, w):
    k = (S.season, w)
    if k not in PAIR_CACHE:
        p = {}
        for g in S.scores[w]:
            m = g.get('metadata') or {}
            if m.get('home_team') and m.get('away_team'): p[m['home_team']] = m['away_team']; p[m['away_team']] = m['home_team']
        PAIR_CACHE[k] = p
    return PAIR_CACHE[k]

def nfl_team(S, pid, w):
    """NFL team in week w: the player's own projection record, else the nearest week that has one (a defense is its own team)."""
    if pid.isalpha(): return pid
    if S.season not in TEAM_HIST:
        th = TEAM_HIST[S.season] = defaultdict(dict)
        for ww, recs in S.proj.items():
            for p, r in recs.items():
                t = r.get('team') or (r.get('player') or {}).get('team')
                if t: th[p][ww] = t
    h = TEAM_HIST[S.season].get(pid)
    if not h: return None
    for d in range(18):
        for ww in (w - d, w + d):
            if ww in h: return h[ww]

def prior_proj(S, key, pid, w):
    for ww in range(w - 1, max(0, w - 5), -1):
        v = S.proj_pts(key, ww).get(pid, 0)
        if v > 0.5: return v
    return 0.0

def build_lineups():
    rows = []
    for L in LEAGUES:
        S = SEASONS_[L['season']]; key = scorer_key(L['league']['scoring_settings']); score = make_scorer(key)
        lid = anon(L['lid']); last = (L['settings'].get('playoff_week_start') or 15) - 1
        for w in range(1, last + 1):
            pp, pair, sdapp, psd = S.proj_pts(key, w), nfl_pair(S, w), S.team_sd(key, L['slots'], w), S.pos_sd(key, w)
            for m in L['matchups'].get(w) or []:
                if not m.get('starters') or m.get('points') is None: continue
                for slot, pid, pts in zip(L['slots'], m['starters'], m['starters_points']):
                    pts = pts or 0.0
                    if pid in (None, '0'):
                        rows.append((lid, L['season'], w, m['roster_id'], m['matchup_id'], sdapp, slot, pid, None, None, None, 'empty', 0.0, 0.0, pts, 0.0, 0.0)); continue
                    pos, team, proj, pl = {'FB': 'RB', 'DB': 'WR'}.get(S.pos(pid), S.pos(pid)), nfl_team(S, pid, w), pp.get(pid, 0.0), S.played(w, pid)   # a few fullbacks / WR-DB hybrids
                    cat = 'played' if pl else ('bye' if team is not None and team not in S.playing[w] else 'dnp')
                    pre = prior_proj(S, key, pid, w) if (cat == 'dnp' and proj <= 0.5) else proj
                    game = '|'.join(sorted([team, pair.get(team) or '?'])) if team else None
                    rows.append((lid, L['season'], w, m['roster_id'], m['matchup_id'], sdapp, slot, pid, pos, team, game, cat, proj, pre, pts,
                                 score(S.stats[w][pid]) if pl else 0.0, psd.get(pos, psd['WR'])))
    d = pd.DataFrame(rows, columns=['lid', 'season', 'week', 'rid', 'mid', 'sd_app', 'slot', 'pid', 'pos', 'nfl', 'game', 'cat', 'proj', 'proj_pre', 'pts', 'pts_model', 'sig_app'])
    d['r'] = d.pts - d.proj
    d['tw'] = d.groupby(['lid', 'week', 'rid'], sort=False).ngroup()
    return d

ST = build_lineups()
print(f"{ST.lid.nunique()} league-seasons, {ST.tw.nunique():,} team-weeks, {len(ST):,} starter slots")
print('starter slots by class:', ST.cat.value_counts().to_dict())
ne = ST[ST.cat != 'empty']
print(f"starters with no known position: {ne.pos.isna().sum()}; played but projected <= 0.5: {((ne.cat=='played')&(ne.proj<=0.5)).sum():,} ({((ne.cat=='played')&(ne.proj<=0.5)).mean():.1%} of non-empty starters)")
print(f"check - league-scored points vs the engine's own scoring of the stat line, played starters: corr {np.corrcoef(ST.pts[ST.cat=='played'], ST.pts_model[ST.cat=='played'])[0,1]:.4f}, "
      f"mean |diff| {np.abs(ST.pts-ST.pts_model)[ST.cat=='played'].mean():.2f} points")

105 league-seasons, 16,988 team-weeks, 167,286 starter slots
starter slots by class: {'played': 163964, 'dnp': 1799, 'empty': 820, 'bye': 703}
starters with no known position: 0; played but projected <= 0.5: 704 (0.4% of non-empty starters)
check - league-scored points vs the engine's own scoring of the stat line, played starters: corr 0.9906, mean |diff| 0.31 points


In [8]:
# team-week table
g = ST.groupby('tw')
TW = g.agg(lid=('lid', 'first'), season=('season', 'first'), week=('week', 'first'), rid=('rid', 'first'), mid=('mid', 'first'), sd_app=('sd_app', 'first'),
           proj=('proj', 'sum'), proj_pre=('proj_pre', 'sum'), pts=('pts', 'sum'), pts_model=('pts_model', 'sum'),
           n_dnp=('cat', lambda c: (c == 'dnp').sum()), n_empty=('cat', lambda c: (c == 'empty').sum()), n_start=('cat', 'size'))
TW['R'] = TW.pts - TW.proj                 # the miss, as specified
TW['R_pre'] = TW.pts - TW.proj_pre         # pregame view: a starter who sat is given his recent projection
TW['R_model'] = TW.pts_model - TW.proj     # actual points re-scored with the engine's scorer (no custom league bonuses)
for c in ('R', 'R_pre', 'R_model'): TW['z' + c[1:]] = TW[c] / TW.sd_app
LSID = {l: i for i, l in enumerate(sorted(TW.lid.unique()))}
TW['ls'] = TW.lid.map(LSID)

# Some leagues use scoring the engine cannot reproduce (extra bonuses, odd settings). Flag them: league points minus the engine's own scoring of the same stat line.
pl_ = ST[ST.cat == 'played']
MAD = (pl_.pts - pl_.pts_model).abs().groupby(pl_.lid).mean()
BAD = set(MAD[MAD > 1.0].index)                        # off by more than 1 point per starter on average
TW['clean'] = ~TW.lid.isin(BAD); ST['clean'] = ~ST.lid.isin(BAD)
print(f"leagues whose scoring the engine misses by > 1 point per starter on average: {len(BAD)} of {len(MAD)} (max {MAD.max():.1f}); "
      f"median miss of the engine scorer {MAD.median():.2f} points per starter; the owner's league {MAD[anon(OWNER_LEAGUE_2025)]:.3f}")

def boot_sd(z, ls, reps=500):
    """Bootstrap by league-season: resample whole leagues and recompute SD(z) and the mean from pooled moments."""
    df = pd.DataFrame({'ls': ls, 'z': z}); a = df.groupby('ls').z.agg(n='size', s='sum', q=lambda v: (v ** 2).sum()).values
    idx = RNG.integers(0, len(a), (reps, len(a))); n, s, q = a[idx, 0].sum(1), a[idx, 1].sum(1), a[idx, 2].sum(1)
    mean = s / n; sd = np.sqrt((q - n * mean ** 2) / (n - 1))
    return np.percentile(sd, [2.5, 97.5]), np.percentile(mean, [2.5, 97.5])

def headline(col, label, mask=None):
    t = TW if mask is None else TW[mask]
    sdci, mci = boot_sd(t[col], t.ls)
    return dict(view=label, leagues=t.lid.nunique(), team_weeks=len(t), mean_miss=t['R' + col[1:]].mean(), sd_miss=t['R' + col[1:]].std(), app_sd=t.sd_app.mean(),
                mean_z=t[col].mean(), sd_z=t[col].std(), sd_z_lo=sdci[0], sd_z_hi=sdci[1])
H = pd.DataFrame([headline('z', '1. as specified: league scoring, all leagues'),
                  headline('z', '2. as specified, leagues the engine can score', TW.clean),
                  headline('z_model', '3. engine-scored actuals, all leagues'),
                  headline('z_pre', '4. pregame view (sitters get recent projection), leagues the engine can score', TW.clean)])
print(H.round(3).to_string(index=False))
SD_Z_ALL, SD_Z, SD_Z_CI, BIAS = H.sd_z[0], H.sd_z[1], (H.sd_z_lo[1], H.sd_z_hi[1]), H.mean_miss[1]
TC = TW[TW.clean]

def by(keys):
    t = TC.groupby(keys, observed=True).agg(n=('z', 'size'), mean_miss=('R', 'mean'), sd_miss=('R', 'std'), app_sd=('sd_app', 'mean'), mean_z=('z', 'mean'), sd_z=('z', 'std'))
    return t.round(3)
print('\nBy season (leagues the engine can score):'); print(by('season'))
TW['wk_bucket'] = pd.cut(TW.week, [0, 2, 4, 8, 12, 99], labels=['1-2', '3-4', '5-8', '9-12', '13+']); TC = TW[TW.clean]
print('\nBy week of the season (the app fits its SDs on the previous four weeks, so the first weeks use fallbacks):'); print(by('wk_bucket'))
print('\nTeam SD in points, by season: real SD of the miss', TC.groupby('season').R.std().round(2).to_dict(), '| app team SD', TC.groupby('season').sd_app.mean().round(2).to_dict())

leagues whose scoring the engine misses by > 1 point per starter on average: 6 of 105 (max 6.3); median miss of the engine scorer 0.15 points per starter; the owner's league 0.033
                                                                         view  leagues  team_weeks  mean_miss  sd_miss  app_sd  mean_z  sd_z  sd_z_lo  sd_z_hi
                                 1. as specified: league scoring, all leagues      105       16988     -1.142   25.280  18.666  -0.055 1.347    1.292    1.432
                                2. as specified, leagues the engine can score       99       15998     -2.272   24.180  18.666  -0.114 1.296    1.283    1.308
                                        3. engine-scored actuals, all leagues      105       16988     -1.290   23.857  18.666  -0.062 1.279    1.265    1.292
4. pregame view (sitters get recent projection), leagues the engine can score       99       15998     -3.254   24.385  18.666  -0.166 1.307    1.294    1.318

By season (leagues the e

### Where does the extra spread come from?

The team's variance is the sum of every starter's own variance, plus twice every pairwise covariance. Four things can make the real number
bigger than the app's:
1. **Starters are not an average player.** The app fits one residual SD per position on everyone projected above 0.5 points; a starter
   is a high-projection player, and high-projection players miss by more (a 25-point QB misses by more than a 3-point WR4).
2. **Teammates move together** (quarterback with his receivers, kicker with his defense).
3. **Opposing sides of one game** (a defense's score is the other offense's failure).
4. **A league-wide weekly factor** (all scoring runs hot or cold relative to projections).

Below, each starter's miss is centred on the average miss of his position, and the team's squared miss is split exactly into these pieces
(pair products summed over all pairs of starters in the lineup). From here on, only the leagues whose scoring the engine can reproduce are used
(99 league-seasons); the six others add scoring noise, not football noise.

In [9]:
SC = ST[ST.clean & (ST.cat != 'empty')].copy()
SC['pos'] = SC.pos.fillna('WR')
SC['c'] = SC.r - SC.groupby('pos').r.transform('mean')
SC['row'] = np.arange(len(SC))
V_POS = SC.groupby('pos').r.var(ddof=0)
print('Residual SD by position, in points: everyone the app calibrates on vs the players actually started')
print(pd.DataFrame({'population (proj > 0.5, played)': PW.groupby('pos').resid.std(), 'starters in these lineups': np.sqrt(V_POS),
                    'avg projection: population': PW.groupby('pos').proj.mean(), 'avg projection: starters': SC[SC.cat == 'played'].groupby('pos').proj.mean()}).round(2))

def cross_by(keys):
    sub = SC.dropna(subset=['nfl'])
    s1 = sub.groupby(keys).c.sum(); s2 = (sub.c ** 2).groupby([sub[k] for k in keys]).sum()
    return (s1 ** 2 - s2).groupby(level='tw').sum()
D = pd.DataFrame({'S2': (SC.c ** 2).groupby(SC.tw).sum(), 'S1': SC.c.groupby(SC.tw).sum()})
D['all'] = D.S1 ** 2 - D.S2
D['same'] = cross_by(['tw', 'nfl']).reindex(D.index).fillna(0)           # 2 * sum of products over same-NFL-team pairs
D['game'] = cross_by(['tw', 'game']).reindex(D.index).fillna(0)
D['opp'] = D.game - D.same                                                # opposite sides of one NFL game
D['other'] = D['all'] - D.game                                            # different games
D['app'] = (TW.sd_app ** 2).reindex(D.index); D['R'] = TW.R.reindex(D.index); D['ls'] = TW.ls.reindex(D.index)
D['tot'] = D.S2 + D.same + D.opp + D.other
COLS = ['app', 'S2', 'same', 'opp', 'other', 'tot']
agg = D.groupby('ls')[COLS].sum().values; n_ls = D.groupby('ls').size().values
bi = RNG.integers(0, len(agg), (500, len(agg))); bs = agg[bi].sum(1); bn = n_ls[bi].sum(1)
def ci(v): return f'[{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]'
DM = D[COLS].mean()
steps = pd.DataFrame({'variance': [DM.app, DM.S2 - DM.app, DM.same, DM.opp, DM.other], 'cumulative': np.cumsum([DM.app, DM.S2 - DM.app, DM.same, DM.opp, DM.other])},
                     index=['what the app assumes: sum of position variances', '+ starters are high-projection players (own spread, no links)',
                            '+ same-NFL-team pairs (QB-WR, K-DEF, ...)', '+ opposite sides of one NFL game', '+ all other pairs (different games: weekly factor)'])
steps['team SD'] = np.sqrt(steps.cumulative); steps['x app SD'] = steps['team SD'] / np.sqrt(DM.app)
print('\nMean team-week variance, built up piece by piece (points^2):'); print(steps.round(3))
VIF = DM.tot / DM.S2
print(f"\nobserved variance of the team miss: {TC.R.var():.1f} (sum of pieces {DM.tot:.1f}); app {DM.app:.1f}")
print(f"VIF = variance with covariances / sum of the starters' own variances = {VIF:.3f}  95% CI {ci(bs[:, 5] / bs[:, 1])}")
print(f"starter-level spread alone: SD x{np.sqrt(DM.S2 / DM.app):.3f} the app's (CI {ci(np.sqrt(bs[:, 1] / bs[:, 0]))}); everything: SD x{np.sqrt(DM.tot / DM.app):.3f} (CI {ci(np.sqrt(bs[:, 5] / bs[:, 0]))})")
print('covariance part by source, as a share of the sum of starter variances:', {k: f'{DM[k] / DM.S2:+.1%}' for k in ('same', 'opp', 'other')},
      '| bootstrap CI same-team:', ci(bs[:, 2] / bs[:, 1]), ' other:', ci(bs[:, 4] / bs[:, 1]))

Residual SD by position, in points: everyone the app calibrates on vs the players actually started
     population (proj > 0.5, played)  starters in these lineups  avg projection: population  avg projection: starters
pos                                                                                                                  
DEF                             5.98                       6.21                        6.54                      7.11
K                               4.77                       4.65                        7.98                      8.54
QB                              7.42                       8.34                       18.66                     21.20
RB                              5.88                       7.19                        7.82                     12.67
TE                              4.80                       6.79                        4.94                     10.04
WR                              6.07                       7.59            

### Same-team pairs in real lineups: how often, and does the step-2 table predict them?

For each pair of started teammates (both projected above 0.5 and played), the table compares the *realised* average product of their centred
misses with the prediction `rho x sigma_i x sigma_j`, taking rho from the position-pair correlation of all same-team players in step 1 and sigma from
the starters' own spread. "Stacks" are lineups that start a quarterback together with at least one of his own receivers (WR or TE).

In [10]:
def pair_rho(df, zcol, team='team'):
    """Correlation of standardized residuals between two players of one NFL team in one week, by position pair (each pair counted once)."""
    d = df.assign(q=df[zcol] ** 2)
    g = d.groupby(['season', 'week', team, 'pos']).agg(s=(zcol, 'sum'), q=('q', 'sum'), n=(zcol, 'size')).unstack('pos').fillna(0)
    out = {}
    for i, a in enumerate(POSITIONS):
        for b in POSITIONS[i:]:
            if a == b: cross, n = ((g['s'][a] ** 2 - g['q'][a]) / 2).sum(), (g['n'][a] * (g['n'][a] - 1) / 2).sum()
            else: cross, n = (g['s'][a] * g['s'][b]).sum(), (g['n'][a] * g['n'][b]).sum()
            out[(a, b)] = out[(b, a)] = (cross / n if n else 0.0, n)
    return out
RHO_ALL = pair_rho(PW, 'z')
A_ = SC[(SC.cat == 'played') & (SC.proj > 0.5) & SC.nfl.notna()][['tw', 'nfl', 'pos', 'c', 'row']]
PR = A_.merge(A_, on=['tw', 'nfl']); PR = PR[PR.row_x < PR.row_y].copy()
PR['type'] = ['-'.join(sorted(t)) for t in zip(PR.pos_x, PR.pos_y)]
PR['pred'] = [RHO_ALL[(a, b)][0] * sqrt(V_POS[a] * V_POS[b]) for a, b in zip(PR.pos_x, PR.pos_y)]
PR['prod'] = PR.c_x * PR.c_y
n_tw = SC.tw.nunique()
pt = PR.groupby('type').agg(pairs=('prod', 'size'), emp_cov=('prod', 'mean'), pred_cov=('pred', 'mean'))
pt['per_lineup'] = pt.pairs / n_tw
pt['var_added_emp'] = 2 * pt.per_lineup * pt.emp_cov; pt['var_added_pred'] = 2 * pt.per_lineup * pt.pred_cov
pt = pt.sort_values('var_added_emp', ascending=False)
print(pt.drop(columns='pairs').round(3).to_string())
N_STACK = PR[PR.type.isin(['QB-WR', 'QB-TE'])].tw.nunique() / n_tw
print(f"\nlineups: {n_tw:,}. Share that start a QB with >= 1 of his own WR/TE: {N_STACK:.1%}. Average same-NFL-team starter pairs per lineup: {len(PR) / n_tw:.2f}"
      f" (QB-with-receiver pairs {pt.per_lineup.get('QB-WR', 0) + pt.per_lineup.get('QB-TE', 0):.2f})")
print(f"variance added by same-team pairs: realised, played pairs only {pt.var_added_emp.sum():.1f}; predicted from the step-2 correlations {pt.var_added_pred.sum():.1f};"
      f" realised, all starter pairs {DM.same:.1f} points^2 per lineup (team variance {TC.R.var():.0f})")

        emp_cov  pred_cov  per_lineup  var_added_emp  var_added_pred
type                                                                
QB-WR    21.907    17.894       0.278         12.168           9.939
QB-TE    16.468    10.515       0.088          2.893           1.847
QB-RB     4.828     3.634       0.141          1.361           1.024
DEF-K    15.105     7.006       0.015          0.453           0.210
K-RB      5.523     2.673       0.033          0.362           0.175
RB-WR     0.541    -0.987       0.333          0.360          -0.657
K-WR      3.597     1.371       0.042          0.299           0.114
DEF-RB    2.807     1.900       0.038          0.213           0.144
TE-TE     4.823     0.555       0.002          0.016           0.002
K-TE      0.345    -0.133       0.015          0.010          -0.004
DEF-TE   -0.073    -2.181       0.020         -0.003          -0.087
K-QB     -3.647     1.706       0.021         -0.156           0.073
RB-RB    -3.774    -1.090       0.

### Starters who did not play

The app fits each position's residual SD only on players who played, so a start who sits (inactive, ruled out on Sunday) is a risk the SD does not
contain: he scores 0 against a positive projection. As explained above, the downloaded projection file has dropped almost every player who did not play,
so in the file view these starters have projection 0 and contribute exactly nothing. The pregame view (recent projection) shows what they would have cost.

In [11]:
NE = ST[ST.clean & (ST.cat != 'empty')]
print('share of non-empty starter slots by class:', NE.cat.value_counts(normalize=True).round(4).to_dict(), '| empty slots:', f'{(ST[ST.clean].cat == "empty").mean():.2%} of all slots')
NB = NE[NE.cat != 'bye']
P_DNP = NB.groupby('pos').cat.apply(lambda c: (c == 'dnp').mean())
print(f"starters whose team played but who did not: {(NB.cat == 'dnp').mean():.2%}; by position:", P_DNP.round(4).to_dict())
dn = NE[NE.cat == 'dnp']
print(f"of {len(dn):,} such starters, projection in the file above 0.5: {(dn.proj > 0.5).sum()}; mean recent (proxy) projection {dn.proj_pre.mean():.1f}; "
      f"lineups with at least one: {(TC.n_dnp > 0).mean():.1%}")
R_dnp = (dn.pts - dn.proj_pre).groupby(dn.tw).sum().reindex(TC.index).fillna(0)
R_pre = TC.R_pre; R_rest = R_pre - R_dnp
v = pd.Series({'whole team miss (pregame view)': R_pre.var(), 'everyone except the sitters': R_rest.var(), 'the sitters alone': R_dnp.var(), '2 x covariance': 2 * np.cov(R_rest, R_dnp)[0, 1]})
print('\nVariance of the team miss, pregame view (points^2):'); print(v.round(1).to_string())
print(f"sitters: {R_dnp.var() / R_pre.var():.1%} of the variance (covariance included {(R_dnp.var() + 2 * np.cov(R_rest, R_dnp)[0, 1]) / R_pre.var():.1%}); mean cost per lineup {R_dnp.mean():.2f} points;"
      f" SD(z) with them {(R_pre / TC.sd_app).std():.3f}, without {(R_rest / TC.sd_app).std():.3f}")
print('in the file view the sitters add exactly 0 (projection 0, score 0): SD(z) =', round(TC.z.std(), 3))
pm = NE[NE.cat == 'played']
print(f"opposite surprise: starters projected <= 0.5 who played: {(pm.proj <= 0.5).mean():.2%} of started players, average {pm[pm.proj <= 0.5].pts.mean():.1f} points scored against a ~0 projection")

share of non-empty starter slots by class: {'played': 0.9862, 'dnp': 0.0099, 'bye': 0.0039} | empty slots: 0.43% of all slots
starters whose team played but who did not: 1.00%; by position: {'DEF': 0.0, 'K': 0.0053, 'QB': 0.0063, 'RB': 0.013, 'TE': 0.0102, 'WR': 0.0105}
of 1,555 such starters, projection in the file above 0.5: 1; mean recent (proxy) projection 10.1; lineups with at least one: 8.3%

Variance of the team miss, pregame view (points^2):
whole team miss (pregame view)    594.6
everyone except the sitters       584.7
the sitters alone                  15.9
2 x covariance                     -5.9
sitters: 2.7% of the variance (covariance included 1.7%); mean cost per lineup -0.98 points; SD(z) with them 1.307, without 1.296
in the file view the sitters add exactly 0 (projection 0, score 0): SD(z) = 1.296
opposite surprise: starters projected <= 0.5 who played: 0.38% of started players, average 0.4 points scored against a ~0 projection


### The matchup: do the two opponents' misses move together?

Win chances depend on **A minus B**. If both teams are hit by the same league-wide scoring weather, that cancels in the difference, so the
spread of the *difference* can be smaller than `sd_A^2 + sd_B^2` suggests. 

In [12]:
Tc = TC[['lid', 'week', 'mid', 'rid', 'R', 'z', 'sd_app', 'season', 'ls']].copy()
Mx = Tc[Tc.groupby(['lid', 'week', 'mid']).R.transform('size') == 2].sort_values(['lid', 'week', 'mid', 'rid'])
Mx['k'] = Mx.groupby(['lid', 'week', 'mid']).cumcount()
MA, MB = Mx[Mx.k == 0].set_index(['lid', 'week', 'mid']), Mx[Mx.k == 1].set_index(['lid', 'week', 'mid'])
tw_of = TC.reset_index()[['tw', 'lid', 'week', 'rid']].set_index(['lid', 'week', 'rid']).tw
tA = tw_of.reindex(list(zip(MA.index.get_level_values(0), MA.index.get_level_values(1), MA.rid))).values     # team-week ids of the two sides
tB = tw_of.reindex(list(zip(MB.index.get_level_values(0), MB.index.get_level_values(1), MB.rid))).values
o_all = corr_ci(MA.R.values, MB.R.values)
RHO_OPP = o_all['r']
wk_mean = Tc.groupby(['season', 'week']).R.transform('mean')                      # same NFL week, all leagues
Mx['R_dm'] = Mx.R - wk_mean.reindex(Mx.index)
o_dm = corr_ci(Mx[Mx.k == 0].R_dm.values, Mx[Mx.k == 1].R_dm.values)
dR = MA.R.values - MB.R.values
print(f"matchups: {len(MA):,}. correlation of the two teams' misses: {o_all['r']:.3f} (95% CI {o_all['lo']:.3f} to {o_all['hi']:.3f});"
      f" after taking out the average miss of that NFL week (all leagues): {o_dm['r']:.3f} ({o_dm['lo']:.3f} to {o_dm['hi']:.3f})")
print(f"var(A-B) = {dR.var():.1f}; var(A) + var(B) = {MA.R.var() + MB.R.var():.1f}; ratio {dR.var() / (MA.R.var() + MB.R.var()):.3f};"
      f" the app assumes var(A-B) = 2 x team SD^2 = {(2 * MA.sd_app ** 2).mean():.1f}  ->  SD ratio {dR.std() / np.sqrt((2 * MA.sd_app.values ** 2).mean()):.3f}")
zd = dR / (np.sqrt(2) * MA.sd_app.values)
print(f"SD of z_diff = (A-B) / (sqrt2 x team SD): {zd.std():.3f}  (what matters for win chances; compare team-level SD(z) {TC.z.std():.3f})")

matchups: 7,999. correlation of the two teams' misses: 0.084 (95% CI 0.062 to 0.106); after taking out the average miss of that NFL week (all leagues): -0.069 (-0.091 to -0.047)
var(A-B) = 1071.0; var(A) + var(B) = 1169.4; ratio 0.916; the app assumes var(A-B) = 2 x team SD^2 = 701.1  ->  SD ratio 1.236
SD of z_diff = (A-B) / (sqrt2 x team SD): 1.244  (what matters for win chances; compare team-level SD(z) 1.296)


## 4. Proposal and prototype

The team SD is clearly too small (step 3), so propose a bigger one. Two ways, from simple to detailed:

* **(a) A global multiplier `k`**: `team SD x k`, with `k` = the spread of z found above. One number, trivially added to `teamSd`.
* **(b) A lineup-aware SD**, built from the pieces measured in step 3:
  `sd_team^2 = sum_i sigma_i^2 + 2 x sum over same-NFL-team starter pairs of rho x sigma_i x sigma_j + c^2 x (sum of projections)^2 + sum_i p_i (1 - p_i) x proj_i^2`.
  The four terms are: each starter's own variance, teammates moving together, a league-wide weekly factor scaled to the team's projection, and the chance a starter
  does not play. Two versions: **(b1)** keeps the app's position SDs for `sigma_i` (the plain covariance fix the task asked for) and **(b2)** lets `sigma_i^2` grow with the starter's
  projection (`a + b x proj`, fitted per position), because step 3 showed that the starters' own spread, not the correlations, is the largest piece.

All parameters (`rho` by position pair, the variance-vs-projection lines, `c^2`, the chance of sitting) are **fitted on two seasons and tested on the third**
(leave-one-season-out), and `k` likewise, so the check below is out-of-sample. Two "views" of the projections are tested: the file as downloaded
(sitters have projection 0, so the sitting term is switched off) and the pregame view (sitters get their recent projection, the sitting term is on).
For win chances, the difference A minus B is what matters; the weekly factor hits both teams, so variant `b2 + cancel` subtracts the part of the
two teams' variances that is shared (`2 c^2 P_A P_B`).

In [13]:
POS_FLOOR = 0.5
def sig2(prm, pos, proj):
    """Variance of a starter's miss: a + b x projection by position, floored at half the position's overall variance."""
    a, b, f = (pos.map({k: v[i] for k, v in prm['var'].items()}) for i in range(3))
    return np.maximum(a + b * proj, f)

def fit_params(train):
    P = PW[PW.season.isin(train)]
    prm = {'var': {}}
    for pos, g in P.groupby('pos'):
        b, a = np.polyfit(g.proj, g.resid ** 2, 1); prm['var'][pos] = (a, b, POS_FLOOR * g.resid.var())
    prm['rho1'] = pair_rho(P, 'z')
    z2 = P.resid / np.sqrt(sig2(prm, P.pos, P.proj)); prm['rho2'] = pair_rho(P.assign(z2=z2 / z2.std()), 'z2')
    nb = SC[SC.season.isin(train) & (SC.cat != 'bye')]
    prm['pdnp'] = nb.groupby('pos').cat.apply(lambda c: (c == 'dnp').mean()).to_dict()
    Q = P[P.proj >= 3]; gq = Q.groupby(['season', 'week'])
    fw = gq.apply(lambda g: g.resid.sum() / g.proj.sum()); nz = gq.apply(lambda g: sig2(prm, g.pos, g.proj).sum() / g.proj.sum() ** 2)
    prm['c2'] = max(fw.var() - nz.mean(), 0.0)                     # variance of the weekly factor (share of projection), noise taken out
    return prm

def make_pairs(view):
    pc = 'proj' if view == 'file' else 'proj_pre'
    a = SC[(SC[pc] > 0.5) & (SC.cat != 'bye') & SC.nfl.notna()][['tw', 'season', 'nfl', 'pos', 'row', 'sig_app', pc]].rename(columns={pc: 'p'})
    m = a.merge(a, on=['tw', 'nfl', 'season']); return m[m.row_x < m.row_y]
PAIRS = {v: make_pairs(v) for v in ('file', 'pregame')}

def components(view, prm, season):
    """Per team-week variance pieces under the parameters `prm`, for the team-weeks of `season`."""
    pc = 'proj' if view == 'file' else 'proj_pre'
    s = SC[(SC.season == season) & (SC[pc] > 0.5) & (SC.cat != 'bye')]
    s2 = sig2(prm, s.pos, s[pc])
    out = pd.DataFrame({'idio2': s2.groupby(s.tw).sum(), 'P': s[pc].groupby(s.tw).sum()})
    out['dnp'] = 0.0
    if view == 'pregame':
        q = s.pos.map(prm['pdnp']); out['dnp'] = (q * (1 - q) * s[pc] ** 2).groupby(s.tw).sum()
    pr = PAIRS[view]; pr = pr[pr.season == season]
    r1 = np.array([prm['rho1'][(a, b)][0] for a, b in zip(pr.pos_x, pr.pos_y)]); r2 = np.array([prm['rho2'][(a, b)][0] for a, b in zip(pr.pos_x, pr.pos_y)])
    sx, sy = np.sqrt(sig2(prm, pr.pos_x, pr.p_x)), np.sqrt(sig2(prm, pr.pos_y, pr.p_y))
    out['pair1'] = pd.Series(2 * r1 * pr.sig_app_x.values * pr.sig_app_y.values, index=pr.tw.values).groupby(level=0).sum()
    out['pair2'] = pd.Series(2 * r2 * (sx * sy).values, index=pr.tw.values).groupby(level=0).sum()
    out = out.fillna(0.0)
    out['c2'] = prm['c2']
    return out

PRM, COMP = {}, {}
for hold in SEASONS_:
    PRM[hold] = fit_params([s for s in SEASONS_ if s != hold])
    for view in ('file', 'pregame'): COMP[(view, hold)] = components(view, PRM[hold], hold)
print('fitted on the other two seasons (tested on the held-out one):')
print(pd.DataFrame({h: {'c (weekly factor, share of projection)': sqrt(p['c2']), 'RB: a + b x proj': '%.1f + %.2f x' % p['var']['RB'][:2], 'WR: a + b x proj': '%.1f + %.2f x' % p['var']['WR'][:2],
                        'QB: a + b x proj': '%.1f + %.2f x' % p['var']['QB'][:2], 'TE: a + b x proj': '%.1f + %.2f x' % p['var']['TE'][:2],
                        'rho QB-WR': p['rho1'][('QB', 'WR')][0], 'rho QB-TE': p['rho1'][('QB', 'TE')][0], 'rho K-DEF': p['rho1'][('K', 'DEF')][0],
                        'P(sits) RB / WR': '%.3f / %.3f' % (p['pdnp']['RB'], p['pdnp']['WR'])} for h, p in PRM.items()}).T.to_string())
print('equivalent weekly-factor correlation for two typical starters (step-2 baseline said ~0.004): c^2 x proj^2 / sigma^2 ~',
      round(float(np.mean([p['c2'] for p in PRM.values()]) * 12 ** 2 / 7 ** 2), 3))

fitted on the other two seasons (tested on the held-out one):
     c (weekly factor, share of projection) RB: a + b x proj WR: a + b x proj QB: a + b x proj TE: a + b x proj rho QB-WR rho QB-TE rho K-DEF P(sits) RB / WR
2023                               0.045126     2.8 + 4.02 x     3.6 + 4.45 x    14.0 + 2.55 x     2.3 + 4.25 x  0.287139   0.18088  0.240069   0.010 / 0.010
2024                               0.039064     2.1 + 4.16 x     1.8 + 4.66 x    22.5 + 2.04 x     3.2 + 3.99 x  0.293013  0.218639  0.243837   0.013 / 0.010
2025                               0.027145     4.5 + 3.88 x     0.3 + 5.11 x    11.5 + 2.51 x     3.1 + 4.05 x  0.267756  0.157129  0.244028   0.016 / 0.011
equivalent weekly-factor correlation for two typical starters (step-2 baseline said ~0.004): c^2 x proj^2 / sigma^2 ~ 0.004


In [14]:
# matchup frame (clean leagues, no ties) and every variant's team variance
TWC = TW[TW.clean].copy()
MU = pd.DataFrame({'tA': tA, 'tB': tB})
for c in ('season', 'ls', 'lid', 'week'): MU[c] = TWC[c].reindex(MU.tA).values
for c in ('pts', 'proj', 'proj_pre', 'sd_app', 'R', 'R_pre'): MU[c + '_A'] = TWC[c].reindex(MU.tA).values; MU[c + '_B'] = TWC[c].reindex(MU.tB).values
MU = MU[MU.pts_A != MU.pts_B].reset_index(drop=True)
MU['yA'] = (MU.pts_A > MU.pts_B).astype(float)
print(f"matchups for calibration: {len(MU):,} (ties dropped: {len(MA) - len(MU)}); side A wins {MU.yA.mean():.1%}")

def team_var(view, kt=None):
    """DataFrame indexed by tw with the variance of each model, fit out-of-sample season by season."""
    parts = []
    for hold in SEASONS_:
        c = COMP[(view, hold)].reindex(TWC.index[TWC.season == hold]).fillna(0.0)
        d = pd.DataFrame(index=c.index); d['app'] = TWC.sd_app.reindex(c.index) ** 2
        d['k'] = d.app * kt[hold] ** 2
        d['L1'] = d.app + c.pair1 + c.c2 * c.P ** 2 + c.dnp
        d['L2'] = c.idio2 + c.pair2 + c.c2 * c.P ** 2 + c.dnp
        d['P'], d['c2'] = c.P, c.c2
        parts.append(d)
    return pd.concat(parts)

def loso_k(zcol_fn):
    return {h: zcol_fn(h) for h in SEASONS_}
KT = loso_k(lambda h: TWC[TWC.season != h].z.std())                                           # team-level multiplier
zd_all = (MU.R_A - MU.R_B) / np.sqrt(MU.sd_app_A ** 2 + MU.sd_app_B ** 2)
KM = loso_k(lambda h: zd_all[MU.season != h].std())                                           # matchup-level multiplier (SD of z_diff)
print('k from the other two seasons - team level:', {h: round(v, 3) for h, v in KT.items()}, '| matchup level (SD of z_diff):', {h: round(v, 3) for h, v in KM.items()})
print('k from all three seasons:', round(TWC.z.std(), 3), 'team;', round(zd_all.std(), 3), 'matchup. By season, team level:', TWC.groupby('season').z.std().round(3).to_dict())
TV = {v: team_var(v, KT) for v in ('file', 'pregame')}

matchups for calibration: 7,997 (ties dropped: 2); side A wins 52.2%
k from the other two seasons - team level: {'2023': 1.302, '2024': 1.298, '2025': 1.274} | matchup level (SD of z_diff): {'2023': 1.232, '2024': 1.251, '2025': 1.249}
k from all three seasons: 1.296 team; 1.245 matchup. By season, team level: {'2023': 1.276, '2024': 1.27, '2025': 1.274}


In [15]:
def nsf(x): return sps.norm.cdf(x)
def fit_slope(x, y, it=8):
    s = 1.0
    for _ in range(it):
        q = 1 / (1 + np.exp(-s * x)); s += (x * (y - q)).sum() / (x * x * q * (1 - q)).sum()
    return s
def scores(p, y):
    P = np.clip(np.r_[p, 1 - p], 1e-9, 1 - 1e-9); Y = np.r_[y, 1 - y]; x = np.log(P / (1 - P))
    return dict(brier=np.mean((P - Y) ** 2), logloss=-np.mean(Y * np.log(P) + (1 - Y) * np.log(1 - P)), slope=fit_slope(x, Y))

MODELS = ['app', 'k team', 'k matchup', 'b1 lineup', 'b2 lineup', 'b2 + cancel']
def matchup_p(view):
    tv = TV[view]; pc = 'proj' if view == 'file' else 'proj_pre'
    vA, vB = tv.loc[MU.tA], tv.loc[MU.tB]; d = (MU[pc + '_A'] - MU[pc + '_B']).values
    var = {'app': vA.app.values + vB.app.values, 'k team': vA.k.values + vB.k.values,
           'k matchup': (vA.app.values + vB.app.values) * np.array([KM[s] for s in MU.season]) ** 2,
           'b1 lineup': vA.L1.values + vB.L1.values, 'b2 lineup': vA.L2.values + vB.L2.values}
    var['b2 + cancel'] = var['b2 lineup'] - 2 * vA.c2.values * vA.P.values * vB.P.values
    return {m: nsf(d / np.sqrt(var[m])) for m in MODELS}, var, d
PM = {v: matchup_p(v) for v in TV}
p_chk = np.array([win_prob(a, b, s) for a, b, s in zip(MU.proj_A[:200], MU.proj_B[:200], MU.sd_app_A[:200])])
assert np.abs(p_chk - PM['file'][0]['app'][:200]).max() < 1e-9, 'vectorised win prob must equal the engine port'

rows = []; BR = {}
reps = 300; ls_rows = [np.where(MU.ls.values == l)[0] for l in np.unique(MU.ls)]
bidx = [np.concatenate([ls_rows[i] for i in RNG.integers(0, len(ls_rows), len(ls_rows))]) for _ in range(reps)]
for view in TV:
    ps, var, d = PM[view]; y = MU.yA.values; tv = TV[view]
    for m in MODELS:
        sc = scores(ps[m], y); b = [scores(ps[m][ix], y[ix]) for ix in bidx]; BR[(view, m)] = b
        zt = None
        if m in ('app', 'k team', 'b1 lineup', 'b2 lineup'):
            col = {'app': 'app', 'k team': 'k', 'b1 lineup': 'L1', 'b2 lineup': 'L2'}[m]
            Rcol = 'R' if view == 'file' else 'R_pre'; zt = TWC[Rcol].reindex(tv.index) / np.sqrt(tv[col])
        zdiff = ((MU.R_A - MU.R_B) if view == 'file' else (MU.R_pre_A - MU.R_pre_B)) / np.sqrt(var[m])
        rows.append(dict(view=view, model=m, sd_z_team=(zt.std() if zt is not None else np.nan),
                         sd_z_2023=(zt[TWC.season.reindex(zt.index) == '2023'].std() if zt is not None else np.nan),
                         sd_z_2024=(zt[TWC.season.reindex(zt.index) == '2024'].std() if zt is not None else np.nan),
                         sd_z_2025=(zt[TWC.season.reindex(zt.index) == '2025'].std() if zt is not None else np.nan),
                         sd_z_diff=zdiff.std(), brier=sc['brier'], logloss=sc['logloss'], slope=sc['slope'],
                         slope_lo=np.percentile([x['slope'] for x in b], 2.5), slope_hi=np.percentile([x['slope'] for x in b], 97.5)))
RES = pd.DataFrame(rows)
print('Out-of-sample check (parameters fitted on the other two seasons). SD of z should be 1; slope 1 = calibrated, < 1 = overconfident.')
print(RES.round(4).to_string(index=False))
def paired(view, m, metric):
    d = np.array([x[metric] - y[metric] for x, y in zip(BR[(view, m)], BR[(view, 'app')])]); return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)
print('\nChange versus the app (negative = better), with 95% bootstrap CI by league-season:')
for view in TV:
    for m in MODELS[1:]:
        b, l = paired(view, m, 'brier'), paired(view, m, 'logloss')
        print(f'  {view:8s} {m:12s} Brier {b[0]:+.5f} [{b[1]:+.5f}, {b[2]:+.5f}]   log loss {l[0]:+.5f} [{l[1]:+.5f}, {l[2]:+.5f}]')

d2 = np.array([x['logloss'] - y['logloss'] for x, y in zip(BR[('file', 'b2 + cancel')], BR[('file', 'k matchup')])])
print(f'lineup-aware b2 + cancel minus simple matchup multiplier, log loss: {d2.mean():+.5f} [{np.percentile(d2, 2.5):+.5f}, {np.percentile(d2, 97.5):+.5f}]  (no real difference if the interval spans 0)')

Out-of-sample check (parameters fitted on the other two seasons). SD of z should be 1; slope 1 = calibrated, < 1 = overconfident.
   view       model  sd_z_team  sd_z_2023  sd_z_2024  sd_z_2025  sd_z_diff  brier  logloss  slope  slope_lo  slope_hi
   file         app     1.2955     1.2762     1.2703     1.2740     1.2445 0.2013   0.5846 0.7798    0.7405    0.8189
   file      k team     1.0030     0.9806     0.9786     1.0001     0.9630 0.1999   0.5796 1.0343    0.9826    1.0835
   file   k matchup        NaN        NaN        NaN        NaN     1.0014 0.1999   0.5795 0.9922    0.9440    1.0412
   file   b1 lineup     1.2380     1.2064     1.2120     1.2347     1.1891 0.2008   0.5826 0.8186    0.7772    0.8576
   file   b2 lineup     1.0158     1.0153     0.9972     0.9807     0.9747 0.1997   0.5789 1.0024    0.9532    1.0494
   file b2 + cancel        NaN        NaN        NaN        NaN     0.9956 0.1997   0.5789 0.9807    0.9326    1.0282
pregame         app     1.3067     1.2963   

In [16]:
# reliability table for the pregame win chance (file view): app vs the best variant
ps = PM['file'][0]; y = MU.yA.values
bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
rel = []
for m in ('app', 'k matchup', 'b2 + cancel'):
    P = np.r_[ps[m], 1 - ps[m]]; Y = np.r_[y, 1 - y]; keep = P >= 0.5
    t = pd.DataFrame({'p': P[keep], 'y': Y[keep]}); t['bin'] = pd.cut(t.p, bins, include_lowest=True)
    g = t.groupby('bin', observed=True).agg(n=('y', 'size'), predicted=('p', 'mean'), observed=('y', 'mean')); g['model'] = m; rel.append(g.reset_index())
REL = pd.concat(rel).pivot(index='bin', columns='model', values=['predicted', 'observed']).round(3)
print('Favourite win chance, predicted vs observed:'); print(REL.to_string())

# chart: z histograms, app vs adjusted
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
xs = np.linspace(-4.5, 4.5, 400)
for ax, (title, z) in zip(axs, [('App: team SD as is', TWC.R.reindex(TV['file'].index) / np.sqrt(TV['file'].app)),
                               ('Global multiplier k (out of sample)', TWC.R.reindex(TV['file'].index) / np.sqrt(TV['file'].k)),
                               ('Lineup-aware SD, b2 (out of sample)', TWC.R.reindex(TV['file'].index) / np.sqrt(TV['file'].L2))]):
    ax.hist(z.clip(-4.5, 4.5), bins=60, range=(-4.5, 4.5), density=True, color='#8fb3cf', edgecolor='white', lw=0.3)
    ax.plot(xs, sps.norm.pdf(xs), color='#c8553d', lw=2, label='N(0,1)')
    ax.set_title(title, fontsize=10, loc='left'); ax.set_xlabel('z = team miss / team SD')
    ax.text(0.97, 0.95, f'SD = {z.std():.2f}\nmean = {z.mean():+.2f}', transform=ax.transAxes, ha='right', va='top', fontsize=9)
axs[0].set_ylabel('density'); axs[0].legend(frameon=False, loc='upper left')
plt.suptitle('Team score misses relative to the team SD (99 league-seasons, weeks before the playoffs)', fontsize=11, x=0.01, ha='left')
plt.tight_layout(); plt.savefig(OUT / 'corr_team_z.png', dpi=140); plt.close()

# what a bigger SD does to a favourite
KTall, KMall = TWC.z.std(), zd_all.std()
conv = pd.DataFrame({'app win chance': [0.55, 0.60, 0.70, 0.80, 0.90]})
conv['x k team (%.2f)' % KTall] = nsf(sps.norm.ppf(conv.iloc[:, 0]) / KTall); conv['x k matchup (%.2f)' % KMall] = nsf(sps.norm.ppf(conv['app win chance']) / KMall)
print('\nEffect on a favourite:'); print(conv.round(3).to_string(index=False))

Favourite win chance, predicted vs observed:
             predicted                       observed                      
model              app b2 + cancel k matchup      app b2 + cancel k matchup
bin                                                                        
(0.499, 0.6]     0.550       0.549     0.549    0.549       0.554     0.554
(0.6, 0.7]       0.648       0.647     0.647    0.615       0.646     0.646
(0.7, 0.8]       0.747       0.747     0.747    0.701       0.737     0.730
(0.8, 0.9]       0.850       0.847     0.847    0.787       0.838     0.852
(0.9, 1.0]       0.954       0.949     0.945    0.921       0.954     0.951

Effect on a favourite:
 app win chance  x k team (1.30)  x k matchup (1.24)
           0.55            0.539               0.540
           0.60            0.578               0.581
           0.70            0.657               0.663
           0.80            0.742               0.751
           0.90            0.839               0.848


In [17]:
# numbers quoted in the summary below
r = lambda name: PT.set_index('pair').loc[name]
print(f"SD(z) app, 99 league-seasons the engine can score: {SD_Z:.3f} (95% CI {SD_Z_CI[0]:.3f} to {SD_Z_CI[1]:.3f}); all 105: {SD_Z_ALL:.3f}; mean miss {BIAS:+.2f} points per team-week; real team SD {TC.R.std():.1f} vs app {TC.sd_app.mean():.1f}")
print(f"VIF {VIF:.3f}; SD multiplier from starter-level spread alone {np.sqrt(DM.S2 / DM.app):.3f}, with covariances {np.sqrt(DM.tot / DM.app):.3f}; pieces of variance: same-team {DM.same / DM.S2:+.1%}, opposite {DM.opp / DM.S2:+.1%}, other {DM.other / DM.S2:+.1%}")
for nme in ('QB1 - WR1', 'QB1 - WR2', 'QB1 - TE1', 'K - DEF', 'QB1 - opp DEF', 'QB1 - opp QB1'):
    x = r(nme); print(f"  {nme}: {x.r:.3f} [{x.lo:.3f}, {x.hi:.3f}] n={x.n}")
print(f"baseline different games: {RHO_WEEK:.3f}; opponents' team misses: {RHO_OPP:.3f}; z_diff SD {zd.std():.3f}; k team {KTall:.3f}, k matchup {KMall:.3f}")
print(f"stacks {N_STACK:.1%} of lineups, {len(PR) / n_tw:.2f} same-team pairs per lineup; sitters {(NB.cat == 'dnp').mean():.2%} of starters, {R_dnp.var() / R_pre.var():.1%} of variance")
print(f'notebook run time (this cell): {time.time() - t_start:.0f}s since the analysis started')

SD(z) app, 99 league-seasons the engine can score: 1.296 (95% CI 1.283 to 1.308); all 105: 1.347; mean miss -2.27 points per team-week; real team SD 24.2 vs app 18.7
VIF 1.089; SD multiplier from starter-level spread alone 1.234, with covariances 1.288; pieces of variance: same-team +2.9%, opposite +0.7%, other +5.3%
  QB1 - WR1: 0.371 [0.328, 0.412] n=1632
  QB1 - WR2: 0.349 [0.306, 0.391] n=1632
  QB1 - TE1: 0.295 [0.250, 0.339] n=1632
  K - DEF: 0.241 [0.194, 0.287] n=1606
  QB1 - opp DEF: -0.406 [-0.446, -0.365] n=1632
  QB1 - opp QB1: 0.180 [0.113, 0.246] n=816
baseline different games: 0.004; opponents' team misses: 0.084; z_diff SD 1.244; k team 1.296, k matchup 1.245
stacks 31.3% of lineups, 1.48 same-team pairs per lineup; sitters 1.00% of starters, 2.7% of variance
notebook run time (this cell): 8s since the analysis started


## 5. What this says

* **The app is overconfident, and not because of correlation alone.** Across 99 league-seasons (6 leagues with scoring the engine cannot reproduce are left out; two more had no
  matchup data), the real miss of a team's lineup has an SD of 24.2 points against the app's 18.7. The ratio SD(z) is **1.30** (95% CI 1.28 to 1.31, bootstrap by league-season) and
  is the same in each season (1.28, 1.27, 1.27). By stretch of the season it runs from 1.17 (weeks 1-2) to 1.41 (week 13 on), noisier but never near 1. The average miss is -2.3 points per team-week (starters score a bit under projection;
  -9.0 in 2025, -0.0 and +1.2 in 2023 and 2024), which does not matter for win chances because both teams are affected.
* **Where the extra spread comes from.** Of the SD ratio of 1.29 that the pieces add up to: **x1.23 is the starters' own spread** (a started RB, WR or TE misses by 6.8 to 7.6 points, not
  the 4.8 to 6.1 the app's population SD implies, because the app averages over everyone projected above 0.5 and starters are the high-projection players; variance grows roughly 4 to 5 points^2 per projected point).
  The correlations add only **VIF = 1.09** (95% CI 1.07 to 1.11) to the variance, about x1.04 on the SD: same-NFL-team pairs +2.9%, opposite sides of a game +0.7%, and a league-wide weekly factor (all other pairs) +5.3%.
* **The correlations are real but they sit in a few pairs, and lineups rarely contain them.** QB1-WR1 0.37 (CI 0.33 to 0.41), QB1-WR2 0.35 (0.31 to 0.39), QB1-TE1 0.30 (0.25 to 0.34), K-DEF 0.24 (0.19 to 0.29);
  a defense against the opposing quarterback -0.41 (-0.45 to -0.37). RB-RB, WR-WR and RB-WR are about 0. Only 31% of lineups start a QB with one of his own receivers and a lineup has 1.5 same-team starter pairs on average,
  so the stack effect adds just 15.6 points^2 to a team variance of 585. Different-game players are almost uncorrelated (0.004), but it is not zero, and it matters through size: the weekly factor behaves like
  about 3 to 4.5% of the team's projected points (fitted per held-out season: 4.5%, 3.9%, 2.7%).
* **Starters who sit hardly matter, and the data hides them.** 1.0% of started players did not play although their team did (8.3% of lineups have one), plus 0.4% byes and 0.4% empty slots. Valued at their recent projection they
  add 2.7% of the team variance and cost about 1 point per lineup on average; SD(z) goes from 1.296 to 1.307. Caveat: the projection file keeps almost only players who played, so in the file as downloaded they cost nothing, and
  this part rests on a proxy.
* **For win chances the right stretch is a little smaller than for the team score: about x1.25.** The two opponents' misses are positively correlated (0.084, CI 0.062 to 0.106) through the weekly factor, so
  var(A-B) is 0.92 of var(A)+var(B), and SD of z for the difference is 1.24. Out of sample, the app's pregame win chance has a calibration slope of 0.78 (CI 0.74 to 0.82), i.e. overconfident (Brier 0.2013, log loss 0.5846).
  A global multiplier fixes it (x1.30 team level: slope 1.03, Brier 0.1999, log loss 0.5796; x1.245 matchup level: slope 0.99, 0.1999, 0.5795), and so does the lineup-aware version b2 (slope 1.00, 0.1997, 0.5789);
  b1, which adds only the covariances and keeps the app's position SDs, barely helps (slope 0.82, 0.2008, 0.5826).
  The simple multiplier and the lineup-aware version are not distinguishable.
* **Recommendation.** Multiply the team SD by about **1.25** for matchup and season simulation (the app already has a `volatility` factor, so the default could simply become 1.25; use 1.30 only if a standalone team score range
  is shown). A 60/40 favourite becomes 58/42, a 70/30 becomes 66/34 and an 80/20 becomes 75/25. If the engine is touched further, the better fix is a starter-level SD that grows with the projection (`sigma^2 = a + b x projection`,
  b about 4 to 5 for RB/WR/TE, about 2 to 2.5 for QB), not a teammate-correlation term; a QB-WR covariance term is worth adding only for exactness.

**Data caveats.** (1) The downloaded projection file has no projection for players who did not play, so sitters are invisible in the file view. (2) Two 2024 league-seasons have empty matchup data and six leagues use scoring
the engine cannot reproduce (one is off by more than 6 points per starter); the all-league SD(z) as specified is 1.35 (CI 1.29 to 1.43). (3) Team weeks 1 to 4 use the app's fallback SDs. (4) Bootstrap intervals resample league-seasons,
but leagues within a season share the same NFL games, so they are somewhat optimistic. (5) Between-season shifts in the average miss (2025 is 9 points low) are real and unmodelled.